### Importing libraries

In [44]:
# %pip install ddgs trafilatura -q

In [45]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display

In [46]:
JUDGE_MODEL = "gpt-4.1-mini"
MODEL = "gpt-4-nano"

### Setup

In [47]:
# api
load_dotenv()

OpenAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key= OpenAI_API_KEY)

if OpenAI_API_KEY is None:
    raise Exception ("No API Key for Open AI")
else:
    print(OpenAI_API_KEY[:8])

sk-proj-


#### Step 1: Define Tools

In [48]:
#search web tool

def search_web(query:str):
    """Seach the web using ddgs browser. Return the top 3 results"""
    #initialize a ddgs object:
    ddgs = DDGS()
    results = ddgs.text(query, max_results = 3)
    print(f'\u2705 Got results')
    #returning in json bc LLM doesnt work with dictionaries, so json converts to text
    return json.dumps(results, indent= 2)

In [49]:
#Fetching the information from the search results. Scraping the url

def fetch_url(url:str):
    """Extracting text from fetched from web result url"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f'\u2705 Got Text {len(text)} characters')
            return text
        else:
            print(f'\u274C Failed to fetch or extract text.')
    return f'Could not extract text from {url}. Try a different source.'

### Step 2: Defining the tools

In [50]:
tools = []

In [51]:
search_web_function = {
    "name" : "search_web",
    "desription" : "Seach the web using ddgs browser. Return the top 3 results",
    "parameters": {
        "type" : "object",
        "properties": {
            "query" : {
                "type" : "string",
                "description" : "The search query to find relevant websites"
            }
            
        },
        "required" : ['query']
    }
}

In [52]:
tools.append({'type' : 'function', 'function': search_web_function})

In [53]:
fetch_url_function = {
    "name" : "fetch_url",
    "description" : "Fetch and extract the main text content from a web page",
    "parameters" : {
        "type" : "object",
        "properties": {
            "url" : {
                "type" : "string",
                "description" : "The url of the webpage to fetch and extract text from."
            }
        },
        "required" : ['url']
    }
}

In [54]:
tools.append({'type' : 'function', 'function': fetch_url_function})

### Testing

In [55]:
# search_web(query= 'AI Healthcare in 2030')
# fetch_url(url = "https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f")
# # fetch_url(url = "https://www.weforum.org/stories/health-and-healthcare-systems/future-of-artificial-intelligence-healthcare-delivery/")


In [56]:
# tools

### Step 3: Tool call handler

In [57]:
def handle_tool_call(tool_calls:list):
        

        tool_call_results = []

        for tool_call in tool_calls:

            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            #  --- for debugging
            print(f" \U0001F527 the function name is {function_name} with arguments {args}")
               
            if function_name == 'search_web':
                result = search_web(args['query']) 

                content = f"Search results : {result}"

            elif function_name == "fetch_url":
                result = fetch_url(args['url'])
                content = f"Fetched URL content : {result}"

            # elif function_name == "insert_function3_name":

            #     content =insert_function3_name (args['message']})

            else:
                 content = f"Unknown function: {function_name}"

            #package into a dictionary for the llm to see what we got out from the tool call
            tool_call_result = {
            'role' : 'tool',
            'content' : content ,
                'tool_call_id' : tool_call.id   
            }

            print(f"this is what the tool_call_result from handle tool_call looks like: {tool_call_result}")

            #appending each tool_call_result to the tool_call_results list of dictionaries
            tool_call_results.append(tool_call_result)

        return tool_call_results

### Step 4: System Prompt

#### This tells the LLM who it is and how to behave. The key things:
* - What its job is
* - What tools it has
* - What process to follow
* - What output format to produe

In [58]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

IMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working trhough deciding what are the
best sources or urls to use. You NEED AT LEAST 6 sources. Make sure to think through which is the best 6 of all your sources. Try and think between tool calls to determine if you need to search the web more for a better topic search
or fetch a different url. You should use your discernment to try and think through the research brief to create a concise 2 paragraph brief, with cited sources. 

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""



### Step 5: Agentic Loop

In [59]:
#Writing the agentic loop for the research agent:
#max_iterations can be anything
def run_research_agent(topic: str, max_iterations: int = 10):
    """ Run the research agent on a topic and return the research brief 

        Args:
            topic: The topic to search
            max_iterations: safely limit to prevent infinte loops    
    """
    print(f'\n\U0001F50D Starting research on : {topic}')
    print('=' * 60)
    #Initialize converstation message list with system prompt and research task
    messages = [
        {'role' : 'system', 'content': RESEARCH_AGENT_PROMPT},
        {'role' : 'user', 'content': f'Research the following topic and produce a comprehensive brief:\n {topic}'}
    ]

    print(f'MESSAGES: BEFORE WHILE LOOP {messages}')
    #LOOP:
    iteration = 0
    while iteration < max_iterations:
        iteration +=1
        print(f'\n--- Iteration {iteration} ---\n')

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model = 'gpt-4.1-mini',
            messages=messages,
            tools = tools
        )
        #extracting the message:
        message = response.choices[0].message
        print(f'This is the message before tool call: {message}')
        #2. Check if LLM called tools
        if message.tool_calls:
            messages.append(message)
            print(f"These are the tool calls: {message.tool_calls}")
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
            print(f'This is the MESSAGES during tool call: {message}')

        #3. Otherwise: no tools were called, read message content
        else:
            content = message.content
            #Check if DONE: , then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f'\n\u2705 Research Complete!')
                return research_brief
            #Otherwise: not yet done
            #loop continues to next iteration
            else:
                print(f'  \U0001F4AD Agent is Thinking:')
                pprint(content[100:])
                

        #4. If we're entering the final iteration, force a final answer
        if(iteration == max_iterations - 1):
            print("  \u26a0 Safety Limit Reached. Stopping research in next iteration")
            messages.append({"role" : "user", "content": "You have reached the maximum number of iterations. You must deliver your research brief now. You MUST respond with DONE: followed by your brief."})

    #Fallback return
    return "Research incomplete. Maximum iterations reached without finalizing brief."



### Let's Run it!

In [60]:
# research_brief = run_research_agent(topic= "AI in healthcare in 2030")
# display(Markdown(research_brief))

### Judge EVAL 1

In [61]:
TOPICS = [
    "Impact of GLP-1 drugs (e.g. Ozempic) on the packaged food industry",
    "Current state of solid-state EV battery commercialization",
    "Effects of remote work on downtown commercial real estate since 2023"]
    
#     ,
#     "Regulatory landscape for AI-generated content and copyright in 2026",
#     "Water scarcity risks to semiconductor manufacturing in Taiwan and Arizona",
#     "Adoption trends of nuclear small modular reactors (SMRs)",
#     "Labor market impact of AI coding assistants on junior software engineer hiring",
#     "Global shift toward de-dollarization in central bank reserves",
#     "Advances in mRNA vaccine platforms beyond COVID-19",
#     "Urban vertical farming economics and scalability",
# ]

In [62]:
JUDGE_PROMPT = """
# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent's **deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** A distinct, identifiable origin of information referenced or drawn upon in the brief (e.g., a named website, publication, report, dataset, database, interview, or document). Count distinct sources, not distinct citations — the same source referenced multiple times still counts once.

**Distinct:** Two references count as separate sources only if they originate from different outlets/authors/publications. Multiple pages, sections, or articles from the same domain or publication count as **one** source unless they are clearly independent publications (e.g., two different news outlets both hosted on the same aggregator would count as two).

**Note:** This check is about breadth of research, not citation formatting. Do not evaluate whether sources are formally cited, footnoted, or linked — only whether the brief's content reflects information drawn from at least 6 distinct sources.

**Source breadth failure (label TRUE):** Any of the below occurred:

1. **Fewer than 6 distinct sources used:** The brief's content, taken as a whole, draws on 5 or fewer distinct sources.
2. **Padded or duplicate sourcing:** The brief appears to reference 6+ sources, but on inspection several are the same underlying source restated, re-fetched, or trivially varied (e.g., same article accessed twice, same domain counted multiple times as if independent).
3. **Single-source dominance:** One source supplies the overwhelming majority of the brief's substantive claims, with other "sources" contributing only trivial or filler information, such that the brief does not reflect genuine breadth.
4. **Unverifiable/fabricated sources counted toward the minimum:** The agent references sources that do not appear to exist or cannot be reasonably inferred as real, and excluding them drops the count below 6.

**No source breadth failure (label FALSE):** The brief's substantive content is genuinely informed by 6 or more distinct, identifiable sources, each contributing non-trivial information to the brief.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations. """

In [63]:
import io, json, contextlib

def run_source_breadth_eval(topics: list[str] = TOPICS, max_iterations: int = 10) -> list[dict]:
    results = []

    for i, topic in enumerate(TOPICS):
        print(f"\n\n========================\n\n[{i+1}/{len(TOPICS)}] Researching: {topic[:60]}...")

        # 1. Run agent
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):  # Suppress print output from the agent for cleaner evaluation logs
            brief = run_research_agent(topic, max_iterations=10)
        if not brief:
            brief = buf.getvalue()

        # 2. Judge
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": brief},
            ],
            max_tokens=1,
            temperature=0,
        )
        verdict = response.choices[0].message.content.strip()
        failed = verdict == "TRUE"

        results.append({"topic": topic, "verdict": verdict, "failed": failed})
        print(f"  → {'FAIL' if failed else 'PASS'}")
        print(brief)

    # --- Summary ---
    n_fail = sum(r["failed"] for r in results)
    print(f"\n{'='*50}")
    print(f"Results: {len(results) - n_fail}/{len(results)} passed  |  {n_fail}/{len(results)} failed")
    print(f"{'='*50}")
    for r in results:
        status = "FAIL ✗" if r["failed"] else "PASS ✓"
        print(f"  [{status}] {r['topic'][:65]}")




In [64]:
results = run_source_breadth_eval()




[1/3] Researching: Impact of GLP-1 drugs (e.g. Ozempic) on the packaged food in...
  → PASS
GLP-1 drugs such as Ozempic and Wegovy, initially developed for type 2 diabetes and more recently approved for obesity treatment, have begun to profoundly impact the packaged food industry by altering consumer eating behavior and spending patterns. These medications reduce appetite and calorie intake, leading to a significant decrease in overall food consumption. Research from Cornell University shows that within six months of initiation, households with a GLP-1 user reduce grocery spending by an average of 5.3%, with higher-income households cutting back by over 8%. Spending on ultra-processed, calorie-dense foods such as savory snacks, sweets, and baked goods declines sharply, while modest increases are noted in healthier food categories like yogurt, fresh fruit, and protein and fiber-rich products. Additionally, users spend about 8% less at fast-food restaurants, coffee shops, and limited-

### Judge EVAL 2

In [65]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

IMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working through deciding what are the
best sources or urls to use.Try and think between tool calls to determine if you need to search the web more for a better topic search
or fetch a different url. You should use your discernment to try and think through the research brief to create a concise 2 paragraph brief, with cited sources. 

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. Try to gather information from 6 sources before delivering your brief. 


When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""



In [66]:
results2 = run_source_breadth_eval()




[1/3] Researching: Impact of GLP-1 drugs (e.g. Ozempic) on the packaged food in...
  → FAIL
GLP-1 drugs like Ozempic, originally developed for type 2 diabetes, have profoundly impacted the packaged food industry by fundamentally altering consumer eating behavior and preferences. These drugs suppress appetite and slow gastric emptying, leading to reduced overall food intake, smaller meal sizes, and lower frequent snacking. As a result, there is a clear shift in food consumption towards nutrient-dense, high-protein, low-sugar, and functional foods that promote satiety and muscle retention during weight loss. Research shows that households with GLP-1 users tend to have smaller grocery baskets but paradoxically spend more on dining out, often opting for smaller portion sizes with high protein content. Major food companies like Danone, Conagra, Nestlé, and Smoothie King have responded by innovating product lines to include high-protein yogurts, protein shakes, fiber-rich snacks, and gut-